# Session 7 — From Diffusion MRI to a Structural Connectome

**Goal of this session:** build one real structural connectome from diffusion MRI data — the physical white-matter wiring, as opposed to the correlation-based functional networks this series has used so far.

*Network Neuroscience in Python, session 7 of 10.*

## Why this matters

Every network in sessions 1 through 6 was either invented outright or built from correlated signals, the same as everything in the first course. Correlation tells you which regions' activity moves together; it says nothing about whether there's an actual bundle of axons physically connecting them. **Diffusion MRI** measures something different and more direct: the tendency of water molecules to diffuse more freely *along* a fibre bundle than *across* it. Follow that directional signal voxel by voxel and you can reconstruct an approximation of the brain's actual white-matter wiring diagram. That reconstruction is what we build in this session, and it is the structural half of session 8's structure-function comparison.

## The dataset

We use **DIPY's Stanford HARDI dataset**: a single anonymised subject's high-angular-resolution diffusion scan, released openly by the Rokem lab at Stanford and bundled with DIPY's own tutorials. `dipy.data.fetch_stanford_hardi()` downloads it — no login, about 90 MB, roughly 25 seconds on a normal connection. It also ships a matching **parcellation** (`fetch_stanford_labels`): a Desikan-Killiany-style atlas already registered to the same diffusion volume, so we don't need a separate registration step to know which voxel belongs to which brain region.

**One subject.** Everything in this notebook describes one person's white matter, not a population. That matters for how much you should read into any specific number below.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt

from dipy.data import read_stanford_labels

img, gtab, labels_img = read_stanford_labels()
data = img.get_fdata()
labels = np.asarray(labels_img.dataobj).astype(int)

print("diffusion volume shape:", data.shape, " (x, y, z, gradient directions)")
print("number of gradient directions (b-vectors):", gtab.bvals.shape[0])
print("parcellation volume shape:", labels.shape)
print("number of distinct labels:", len(np.unique(labels)))

## From raw diffusion signal to fibre orientation

Two steps turn the raw signal into something we can track through:

1. **Tensor fitting.** The simplest model of diffusion at each voxel is a single 3D ellipsoid (a "tensor") describing how far water moved in every direction. From it we get **fractional anisotropy (FA)**: 0 means diffusion is equally free in every direction (grey matter, cerebrospinal fluid — no strong fibre direction), close to 1 means diffusion is strongly directional (a well-organised, single-direction white-matter bundle). We use FA only as a stopping rule for tracking, not to decide fibre direction — a single ellipsoid can't represent two fibre bundles crossing in the same voxel, which happens constantly in real white matter.

2. **Constrained spherical deconvolution (CSD).** A more flexible model that can recover *multiple* fibre orientations per voxel, built to handle exactly the crossing-fibres problem tensors can't. We fit it once per white-matter voxel and keep its peak directions — the estimated fibre orientation(s) passing through that voxel.

Both of these run on a subset of the volume already restricted to a white-matter mask, to keep runtime sane.

In [ ]:
from dipy.reconst.dti import TensorModel, fractional_anisotropy
from dipy.reconst.csdeconv import ConstrainedSphericalDeconvModel, auto_response_ssst
from dipy.direction import peaks_from_model
from dipy.data import default_sphere

white_matter = (labels == 1) | (labels == 2)  # cerebral WM + corpus callosum, from the atlas

tenfit = TensorModel(gtab).fit(data, mask=white_matter)
fa = fractional_anisotropy(tenfit.evals)
fa[np.isnan(fa)] = 0
print(f"FA in white matter: mean={fa[white_matter].mean():.2f}, "
      f"range [{fa[white_matter].min():.2f}, {fa[white_matter].max():.2f}]")

response, ratio = auto_response_ssst(gtab, data, roi_radii=10, fa_thr=0.7)
print(f"estimated single-fibre response function, ratio={ratio:.3f}")

## The heaviest computation in this series — already run for you

Fitting CSD across the whole white-matter mask and then tracking tens of thousands of streamlines through it takes about a minute on a laptop CPU — fine to run once, not something to sit through live on a recorded call, and Colab's free-tier CPU allocation can be slower still. So `scripts/build_structural_connectome.py` in this repo does exactly that, once, and saves three small files to `data/`:

- `structural_connectome.npy` — an 86 × 86 matrix of streamline counts between region pairs
- `structural_region_labels.csv` — the name and hemisphere of each of the 86 regions
- `structural_sample_streamlines.npz` — 300 individual streamlines, kept only for the plot below

From here on, we load and plot. If you want to see the full pipeline (CSD fitting, deterministic tracking with `LocalTracking`, building the matrix with `dipy.tracking.utils.connectivity_matrix`), read that script — it's short, and it's the same code that produced these files.

In [ ]:
import io
import os
import urllib.request

REPO_RAW = "https://raw.githubusercontent.com/saeedrafsharx/network-neuroscience-python/main/data/"
DATA = "../data/" if os.path.exists("../data/structural_connectome.npy") else REPO_RAW
print("reading cached data from:", DATA)


def load_array(name, npz=False):
    if DATA.startswith("http"):
        with urllib.request.urlopen(DATA + name) as response:
            buf = io.BytesIO(response.read())
    else:
        buf = DATA + name
    return np.load(buf, allow_pickle=True) if npz else np.load(buf)


connectome = load_array("structural_connectome.npy")
streams = load_array("structural_sample_streamlines.npz", npz=True)
sample_streamlines = streams["streamlines"]

import pandas as pd
region_labels = pd.read_csv(DATA + "structural_region_labels.csv")

print("connectome:", connectome.shape)
print("sample streamlines:", len(sample_streamlines))
region_labels.head()

## A handful of tracked streamlines

Real tractography produces tens of thousands of individual fibre reconstructions through the full 3D volume. Plotting all of them would just be visual noise on a recorded screen, so here are 300 of them projected onto a single 2D plane (looking down the z-axis) — enough to see that they follow coherent, elongated bundle shapes rather than random walks.

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 8))
for sl in sample_streamlines[:300]:
    ax.plot(sl[:, 0], sl[:, 1], linewidth=0.5, alpha=0.5, color="#2b6cb0")
ax.set_xlabel("x (mm)", fontsize=12)
ax.set_ylabel("y (mm)", fontsize=12)
ax.set_title("300 tracked streamlines, top-down projection", fontsize=13)
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

## The structural connectivity matrix

Row *i*, column *j* is the number of tracked streamlines that start in region *i* and end in region *j* (or vice versa — connectivity here is undirected, since deterministic tractography can't reliably tell you which end is the "start"). This is on a genuinely different scale to the correlation matrices from the first course: it's a literal count of reconstructed fibre bundles, not a similarity score, and it is heavily right-skewed — most region pairs share zero or very few streamlines, and a handful of pairs share a great many.

In [ ]:
print(f"total streamline count across all region pairs: {int(connectome.sum())}")
print(f"nonzero region pairs: {int((connectome > 0).sum())} / {connectome.size} "
      f"({100 * (connectome > 0).mean():.1f}%)")

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(np.log1p(connectome), cmap="viridis")
ax.set_title("Structural connectome (log(1 + streamline count))", fontsize=13)
ax.set_xlabel("region index", fontsize=12)
ax.set_ylabel("region index", fontsize=12)
fig.colorbar(im, ax=ax, label="log(1 + streamlines)", shrink=0.8)
plt.tight_layout()
plt.show()

## What this matrix is, and is not

This is one person's estimated white-matter wiring, built with a widely-used but genuinely imperfect method. Deterministic tractography is known to both miss real fibre pathways and hallucinate implausible ones — reconstructing a bundle where the true anatomy doesn't support one — and different tractography algorithms run on the exact same data can disagree substantially on the details. Treat this matrix as a reasonable, standard-pipeline estimate of structural connectivity, not as ground truth. That honesty matters most in session 8, where we'll compare it to a functional network and be tempted to over-interpret how closely the two agree.

**Next session:** put this structural connectome next to a functional one, edge by edge, and talk honestly about what the relationship between them actually is — and isn't.